# Diagnóstico de rutas en Google Drive
Ejecutar cada celda en orden para encontrar dónde quedaron los archivos.

In [1]:
# CELDA 1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive montado OK')

Mounted at /content/drive
Drive montado OK


In [2]:
# CELDA 2 — Buscar el archivo nasa_power por todo el Drive
import subprocess

print('Buscando nasa_power*.csv en todo el Drive...')
resultado = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', 'nasa_power*.csv'],
    capture_output=True, text=True, timeout=60
)
if resultado.stdout.strip():
    print('ENCONTRADO en:')
    for linea in resultado.stdout.strip().split('\n'):
        print(' ', linea)
else:
    print('NO encontrado. Buscando por nombre parcial...')
    resultado2 = subprocess.run(
        ['find', '/content/drive/MyDrive', '-name', '*nasa*'],
        capture_output=True, text=True, timeout=60
    )
    print(resultado2.stdout if resultado2.stdout else 'Tampoco encontrado con *nasa*')

Buscando nasa_power*.csv en todo el Drive...
ENCONTRADO en:
  /content/drive/MyDrive/Maestria Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Codigos Trabajo de Grado/datos/nasa_power_pasto_2013_2026_horario.csv


In [3]:
# CELDA 3 — Listar el contenido de la carpeta del proyecto
import os

# Buscar la carpeta 'Codigos Trabajo de Grado' sin importar tildes
resultado3 = subprocess.run(
    ['find', '/content/drive/MyDrive', '-type', 'd', '-name', '*Trabajo de Grado*'],
    capture_output=True, text=True, timeout=60
)
print('Carpetas encontradas con "Trabajo de Grado" en el nombre:')
carpetas = resultado3.stdout.strip().split('\n')
for c in carpetas:
    if c:
        print(' ', c)

Carpetas encontradas con "Trabajo de Grado" en el nombre:
  /content/drive/MyDrive/Maestría Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Códigos Trabajo de Grado
  /content/drive/MyDrive/Maestria Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Codigos Trabajo de Grado


In [4]:
# CELDA 4 — Listar TODO el contenido de esa carpeta (incluyendo subcarpetas)
# Reemplazar la ruta con la que apareció en la celda anterior

# La ruta correcta aparecerá en la CELDA 3 — pégala aquí:
RUTA_PROYECTO = carpetas[0] if carpetas and carpetas[0] else '/content/drive/MyDrive'

print(f'Listando contenido de: {RUTA_PROYECTO}')
print()

for raiz, dirs, archivos in os.walk(RUTA_PROYECTO):
    nivel = raiz.replace(RUTA_PROYECTO, '').count(os.sep)
    sangria = '  ' * nivel
    print(f'{sangria}[DIR] {os.path.basename(raiz)}/')
    subsangria = '  ' * (nivel + 1)
    for archivo in archivos:
        ruta_completa = os.path.join(raiz, archivo)
        tam = os.path.getsize(ruta_completa) / 1024  # KB
        print(f'{subsangria}{archivo}  ({tam:.1f} KB)')

Listando contenido de: /content/drive/MyDrive/Maestría Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Códigos Trabajo de Grado

[DIR] Códigos Trabajo de Grado/
  Datos_2013_2023.xlsx  (8720.4 KB)
  Datos_2013_2026_completo.csv  (31099.1 KB)
  Datos_2013_2026_completo.gsheet  (0.2 KB)
  00_descarga_nasa_power_v2.ipynb  (271.5 KB)
  00_diagnostico_rutas.ipynb  (5.2 KB)


In [5]:
# CELDA 5 — Mostrar la ruta EXACTA que debes usar en los notebooks
print('=' * 60)
print('RUTA EXACTA PARA USAR EN LOS NOTEBOOKS:')
print('=' * 60)
print()
print(f'RUTA_PROYECTO = "{RUTA_PROYECTO}"')
print()
print('Copiar esta línea en todos los notebooks (celda de Drive):')
print(f'RUTA_BASE = "{RUTA_PROYECTO}"')

RUTA EXACTA PARA USAR EN LOS NOTEBOOKS:

RUTA_PROYECTO = "/content/drive/MyDrive/Maestría Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Códigos Trabajo de Grado"

Copiar esta línea en todos los notebooks (celda de Drive):
RUTA_BASE = "/content/drive/MyDrive/Maestría Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Códigos Trabajo de Grado"


In [6]:
# CELDA 6 — Si el CSV de NASA POWER fue encontrado, cargarlo para verificar
import pandas as pd

resultado_nasa = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', 'nasa_power*.csv'],
    capture_output=True, text=True, timeout=60
)

if resultado_nasa.stdout.strip():
    ruta_nasa = resultado_nasa.stdout.strip().split('\n')[0]
    print(f'Cargando: {ruta_nasa}')
    df = pd.read_csv(ruta_nasa, index_col=0, parse_dates=True, nrows=5)
    print(f'OK — {df.shape[1]} columnas, primeras filas:')
    print(df)
else:
    print('El CSV de NASA POWER NO existe en Drive.')
    print()
    print('Causa probable: el notebook 00 creo la carpeta "datos" en')
    print('una ruta diferente a la que ves en Drive.')
    print()
    print('Solucion: copiar la RUTA_BASE de la Celda 5 y reejecutar el notebook 00.')

Cargando: /content/drive/MyDrive/Maestria Ciencia de Datos Segundo Semestre/Proyecto Aplicado ll/Codigos Trabajo de Grado/datos/nasa_power_pasto_2013_2026_horario.csv
OK — 10 columnas, primeras filas:
                     ALLSKY_SFC_SW_DWN  CLRSKY_SFC_SW_DWN    T2M   RH2M  \
timestamp_utc                                                             
2013-08-01 00:00:00                0.0                0.0  10.68  91.51   
2013-08-01 01:00:00                0.0                0.0  10.29  94.93   
2013-08-01 02:00:00                0.0                0.0   9.92  96.26   
2013-08-01 03:00:00                0.0                0.0   9.53  96.59   
2013-08-01 04:00:00                0.0                0.0   9.13  96.17   

                     PRECTOTCORR  WS10M  WD10M     PS   SZA   KT  
timestamp_utc                                                     
2013-08-01 00:00:00         1.38   1.46  143.9  75.33  90.0  0.0  
2013-08-01 01:00:00         0.81   1.68  139.1  75.37  90.0  0.0  
2013-